# Africapolis Density Scenario Shapefiles — 探索
运行每个 cell，结果就地显示。目标是搞清楚：
- 文件结构和命名规律
- 每个 shapefile 的字段（columns）
- geometry 类型和坐标系
- 与 nodes 数据的连接键
- 三种 density scenario 的具体差异

In [ ]:
import os
import glob
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── 修改这里：你的 Density_Scenario_Paper 路径 ──────────────────────────
BASE_DIR = "input/Density_Scenario_Paper"
# ────────────────────────────────────────────────────────────────────────

print(f"Looking in: {os.path.abspath(BASE_DIR)}")
print(f"Exists: {os.path.exists(BASE_DIR)}")

## 1. 目录结构总览

In [ ]:
# 列出所有子目录和文件
all_files = []
for root, dirs, files in os.walk(BASE_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))

# 按扩展名统计
exts = pd.Series([os.path.splitext(f)[1].lower() for f in all_files])
print("=== 文件扩展名统计 ===")
print(exts.value_counts().to_string())
print(f"\n总文件数: {len(all_files)}")

In [ ]:
# 列出所有 .shp 文件
shp_files = sorted(glob.glob(os.path.join(BASE_DIR, "**/*.shp"), recursive=True))
print(f"共找到 {len(shp_files)} 个 shapefile:\n")
for f in shp_files:
    rel = os.path.relpath(f, BASE_DIR)
    size_kb = os.path.getsize(f) / 1024
    print(f"  {rel:60s}  {size_kb:8.1f} KB")

In [ ]:
# 解析文件名规律：区域、国家、scenario
records = []
for f in shp_files:
    parts = os.path.normpath(f).split(os.sep)
    basename = os.path.splitext(os.path.basename(f))[0]
    # 找到 BASE_DIR 之后的层级
    rel_parts = parts[len(os.path.normpath(BASE_DIR).split(os.sep)):]
    records.append({
        "path": f,
        "subfolder": rel_parts[0] if len(rel_parts) > 1 else "",
        "basename": basename,
        "depth": len(rel_parts),
    })

df_files = pd.DataFrame(records)
print("=== 子文件夹 ===")
print(df_files["subfolder"].value_counts().to_string())
print("\n=== basename 样本 ===")
print(df_files["basename"].head(20).to_string())

## 2. 读取一个样本 shapefile，检查字段

In [ ]:
# 读第一个 shapefile 作为样本
sample_path = shp_files[0]
print(f"Sample: {os.path.relpath(sample_path, BASE_DIR)}\n")

gdf = gpd.read_file(sample_path)
print(f"Shape: {gdf.shape}")
print(f"CRS: {gdf.crs}")
print(f"Geometry types: {gdf.geometry.geom_type.value_counts().to_dict()}")
print(f"\nColumns:")
for col in gdf.columns:
    dtype = gdf[col].dtype
    n_unique = gdf[col].nunique() if col != 'geometry' else '-'
    sample_val = gdf[col].iloc[0] if col != 'geometry' else f"<{gdf.geometry.geom_type.iloc[0]}>"
    print(f"  {col:30s} dtype={str(dtype):12s} nunique={str(n_unique):6s} sample={sample_val}")

In [ ]:
# 显示前5行（非geometry列）
non_geom = [c for c in gdf.columns if c != 'geometry']
gdf[non_geom].head()

In [ ]:
# 统计信息
print("=== 数值列统计 ===")
print(gdf[non_geom].describe().to_string())

## 3. 检查所有 shapefile 的字段一致性

In [ ]:
# 读所有 shapefiles 的 columns + CRS（不读geometry内容，只读schema）
import fiona

schema_records = []
for f in shp_files:
    try:
        with fiona.open(f) as src:
            schema_records.append({
                "file": os.path.relpath(f, BASE_DIR),
                "n_features": len(src),
                "crs": str(src.crs),
                "geom_type": src.schema["geometry"],
                "fields": list(src.schema["properties"].keys()),
                "field_types": list(src.schema["properties"].values()),
            })
    except Exception as e:
        schema_records.append({"file": f, "error": str(e)})

df_schema = pd.DataFrame(schema_records)
print(f"Total files read: {len(df_schema)}")
print(f"\nGeometry types:")
print(df_schema['geom_type'].value_counts().to_string())
print(f"\nCRS:")
print(df_schema['crs'].value_counts().to_string())
print(f"\nn_features range: {df_schema['n_features'].min()} – {df_schema['n_features'].max()}")

In [ ]:
# 检查字段是否在所有文件中一致
field_sets = [frozenset(r['fields']) for r in schema_records if 'fields' in r]
unique_field_sets = set(field_sets)
print(f"Unique field schemas: {len(unique_field_sets)}")

if len(unique_field_sets) == 1:
    print("✅ 所有文件字段完全一致")
    print("Fields:", sorted(list(unique_field_sets)[0]))
else:
    print("⚠️  字段不一致，各组如下：")
    for i, fs in enumerate(unique_field_sets):
        files_with_this = [r['file'] for r in schema_records 
                           if 'fields' in r and frozenset(r['fields']) == fs]
        print(f"\nGroup {i+1} ({len(files_with_this)} files): {sorted(fs)}")
        print("  Files:", files_with_this[:5], "..." if len(files_with_this) > 5 else "")

In [ ]:
# 完整 schema 表格
df_schema[['file','n_features','geom_type','crs','fields']]

## 4. 三种 Density Scenario 的区别

In [ ]:
# 尝试识别 scenario 类型：从文件名或字段中寻找 high/consolidated/low 的痕迹
for f in shp_files:
    basename = os.path.basename(f).lower()
    if any(k in basename for k in ['high','consol','low','hd','cd','ld']):
        print(f"Scenario keyword in filename: {os.path.relpath(f, BASE_DIR)}")

# 检查字段名中是否有 scenario 相关信息
if schema_records and 'fields' in schema_records[0]:
    print("\nAll fields in first file:", schema_records[0]['fields'])

In [ ]:
# 如果同一个国家有多个 shapefile，对比它们
# 提取国家代码（假设basename格式如 CAF100002）
import re

for rec in schema_records:
    basename = os.path.splitext(os.path.basename(rec['file']))[0]
    # 尝试匹配 3字母国家代码 + 数字
    m = re.match(r'^([A-Z]{2,3})(\d+)', basename)
    rec['country_code'] = m.group(1) if m else basename[:3]
    rec['file_num'] = m.group(2) if m else ''

df_schema2 = pd.DataFrame(schema_records)
print("Country codes found:")
print(df_schema2['country_code'].value_counts().to_string())

print("\nFiles per country (>1 might indicate multiple scenarios per country):")
multi = df_schema2.groupby('country_code').size()
print(multi[multi > 1].to_string())

## 5. 可视化：一个国家的所有 polygons

In [ ]:
# 选一个国家，把它的所有 shapefile 叠加画出来
FOCUS_COUNTRY = "NIG"   # ← 可以改成你想看的区域/国家文件夹

focus_files = [f for f in shp_files if FOCUS_COUNTRY in f]
print(f"Files for '{FOCUS_COUNTRY}': {len(focus_files)}")
for f in focus_files:
    print(" ", os.path.relpath(f, BASE_DIR))

In [ ]:
# 读取并叠加显示（如果有多个文件则分色）
if focus_files:
    gdfs = []
    for f in focus_files:
        g = gpd.read_file(f)
        g['source_file'] = os.path.basename(f)
        gdfs.append(g)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    
    for i, (g, f) in enumerate(zip(gdfs, focus_files)):
        g.plot(ax=ax, alpha=0.4, color=colors[i % len(colors)],
               edgecolor='black', linewidth=0.5,
               label=os.path.basename(f))
    
    ax.set_title(f"Density scenario polygons: {FOCUS_COUNTRY}")
    ax.legend(fontsize=8)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.tight_layout()
    plt.show()
    
    # Print bounding boxes
    for g, f in zip(gdfs, focus_files):
        bounds = g.total_bounds  # [minx, miny, maxx, maxy]
        print(f"{os.path.basename(f)}: bbox={bounds.round(3)}, n={len(g)}")

## 6. 连接键：polygon 如何对应到 nodes？

In [ ]:
# 读 nodes 数据
nodes = pd.read_csv("input/AfricaNetworkNodes.csv")
afp   = pd.read_csv("input/Africapolis_2050.csv")

print("Nodes columns:", list(nodes.columns))
print("Africapolis columns:", list(afp.columns[:15]), "...")

# 读一个 shapefile，看它的字段
gdf_s = gpd.read_file(shp_files[0])
print(f"\nSample shapefile ({os.path.basename(shp_files[0])}) columns:")
print([c for c in gdf_s.columns if c != 'geometry'])

In [ ]:
# 检查 shapefile 字段值，寻找与 Agglomeration_ID 或 agglosName 的交集
non_geom_cols = [c for c in gdf_s.columns if c != 'geometry']

for col in non_geom_cols:
    vals = set(gdf_s[col].dropna().astype(str))
    
    # Check overlap with node names
    node_names = set(nodes['agglosName'].dropna().astype(str))
    node_ids   = set(nodes['name'].dropna().astype(str))
    afp_ids    = set(afp['Agglomeration_ID'].dropna().astype(str))
    afp_names  = set(afp['Agglomeration_Name'].dropna().astype(str))
    
    overlap_name = len(vals & node_names)
    overlap_id   = len(vals & node_ids)
    overlap_afp_id   = len(vals & afp_ids)
    overlap_afp_name = len(vals & afp_names)
    
    if any(x > 0 for x in [overlap_name, overlap_id, overlap_afp_id, overlap_afp_name]):
        print(f"  Column '{col}':")
        print(f"    ↔ nodes.agglosName overlap: {overlap_name}")
        print(f"    ↔ nodes.name overlap:       {overlap_id}")
        print(f"    ↔ Africapolis ID overlap:   {overlap_afp_id}")
        print(f"    ↔ Africapolis Name overlap: {overlap_afp_name}")
        print(f"    Sample values: {list(vals)[:5]}")

In [ ]:
# 全量检查：所有 shapefile 的字段样本
print("=== 每个 shapefile 的字段和前3行 ===")
for f in shp_files:
    try:
        g = gpd.read_file(f, rows=3)
        nc = [c for c in g.columns if c != 'geometry']
        print(f"\n{os.path.relpath(f, BASE_DIR)}")
        print(g[nc].to_string(index=False))
    except Exception as e:
        print(f"  ERROR: {e}")

## 7. 面积分析：三种 scenario 的 polygon 大小对比

In [ ]:
# 读所有 shapefile，计算面积（先统一投影到等积坐标系）
all_gdfs = []
for f in shp_files:
    try:
        g = gpd.read_file(f)
        g['source_file'] = os.path.basename(f)
        g['region'] = os.path.basename(os.path.dirname(f))  # 子文件夹名
        all_gdfs.append(g)
    except Exception as e:
        print(f"Error reading {f}: {e}")

if all_gdfs:
    gdf_all = pd.concat(all_gdfs, ignore_index=True)
    print(f"Total polygons loaded: {len(gdf_all)}")
    print(f"Columns: {[c for c in gdf_all.columns if c != 'geometry']}")
    
    # Project to Africa Albers Equal Area for area calculation
    gdf_proj = gdf_all.to_crs("+proj=aea +lat_1=20 +lat_2=-23 +lat_0=0 +lon_0=25")
    gdf_all['area_km2'] = gdf_proj.geometry.area / 1e6
    
    print("\nArea stats by region folder:")
    print(gdf_all.groupby('region')['area_km2'].describe().round(1).to_string())

In [ ]:
# 如果 shapefile 字段里有 scenario 标识，分组对比面积
# （根据上面发现的字段名替换 'SCENARIO_COL'）
SCENARIO_COL = None   # ← 填入你在上面发现的场景字段名，如 'scenario' 或 'density'

if SCENARIO_COL and SCENARIO_COL in gdf_all.columns:
    print(f"Area by scenario ('{SCENARIO_COL}'):")
    print(gdf_all.groupby(SCENARIO_COL)['area_km2']
          .agg(['count','mean','median','sum']).round(1).to_string())
    
    fig, ax = plt.subplots(figsize=(8,4))
    for sc in gdf_all[SCENARIO_COL].unique():
        subset = gdf_all[gdf_all[SCENARIO_COL] == sc]['area_km2']
        subset[subset < subset.quantile(0.99)].hist(ax=ax, bins=50, alpha=0.5, label=str(sc))
    ax.set_xlabel('Area (km²)')
    ax.set_title('Urban polygon area distribution by density scenario')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("SCENARIO_COL not set — fill it in based on what you found above.")

## 8. 小结：请记录以下信息，告诉 Claude

运行完上面所有 cell 后，请截图或复制以下信息：

1. **Section 1**：shapefile 总数 + 子文件夹列表
2. **Section 2**：样本 shapefile 的全部字段名
3. **Section 3**：字段是否一致？有哪几种字段组合？
4. **Section 4**：三种 scenario 是分文件还是同一文件的不同列/字段？
5. **Section 6**：哪个字段可以用来连接 nodes/Africapolis？
6. **Section 7（如果 Section 4 找到了 SCENARIO_COL）**：三种 scenario 的面积差异大致是多少？